# MLP、LSTM 与 TFT 统一比较

只读取当前输入哈希匹配的最新三组独立运行结果。

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
from pathlib import Path
import os, sys, hashlib


def find_repo_root():
    configured = os.environ.get("RETAIL_FORECAST_ROOT")
    candidates = (
        [Path(configured).expanduser()]
        if configured
        else [Path.cwd(), *Path.cwd().parents]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "notebooks"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT"
    )


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from retail_forecast.common import *
from retail_forecast.model_comparison import latest_compatible

In [ ]:
models = {"MLP": "neural_mlp", "LSTM": "neural_lstm", "TFT": "neural_tft"}
sources = {model: latest_compatible(directory) for model, directory in models.items()}
metrics_tables = [pd.read_csv(path / "metrics.csv") for path in sources.values()]
prediction_tables = [
    pd.read_csv(path / "predictions.csv", parse_dates=["ds", "forecast_origin"])
    for path in sources.values()
]
explanation_tables = [
    pd.read_csv(path / "explanations.csv") for path in sources.values()
]
neural_metrics = pd.concat(metrics_tables, ignore_index=True)
neural_predictions = pd.concat(prediction_tables, ignore_index=True)
neural_explanations = pd.concat(explanation_tables, ignore_index=True)
display(
    neural_metrics.groupby(["scenario", "model", "split"], as_index=False)[
        ["MAE", "RMSE", "WAPE", "Bias"]
    ]
    .mean()
    .round(4)
)

In [ ]:
out = RUNS_DIR / "neural_comparison"
out.mkdir(parents=True, exist_ok=False)
summary = neural_metrics.groupby(["scenario", "model", "split"], as_index=False)[
    ["MAE", "RMSE", "WAPE", "Bias"]
].mean()
summary.to_csv(out / "model_comparison.csv", index=False)
neural_predictions.to_csv(out / "predictions.csv", index=False)
neural_explanations.to_csv(out / "explanations.csv", index=False)
import json

(out / "manifest.json").write_text(
    json.dumps(
        {
            "family": "neural_comparison",
            "models": list(models),
            "protocol_version": "retail-benchmark-v2",
            "hashes": INPUT_HASHES,
        },
        indent=2,
    )
)
fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(22, 4), constrained_layout=True)
for ax, scenario in zip(axes, SCENARIOS):
    group = neural_predictions.loc[
        (neural_predictions.scenario == scenario) & neural_predictions.split.eq("test")
    ]
    first = group.loc[group.model.eq("MLP")]
    ax.plot(first.ds, first.y, "ko-", ms=3, label="Observed")
    for model in models:
        values = group.loc[group.model.eq(model)]
        ax.plot(values.ds, values.prediction, label=model)
    ax.set(title=scenario)
    ax.legend()
fig.savefig(out / "neural_test_predictions.png", dpi=150)
plt.show()
plt.close(fig)